In [ ]:
import numpy as np
import torch
from torch.amp.autocast_mode import autocast
from scipy.spatial import KDTree
import time
import json
import os
import hashlib
from gensim.models import Word2Vec

# Функции для вычисления хэша
def compute_file_hash(file_path):
    """Вычисляет MD5-хэш одного файла."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def compute_dataset_hash(files):
    """Вычисляет общий хэш набора данных на основе хэшей отдельных файлов."""
    hashes = sorted(compute_file_hash(f) for f in files if os.path.exists(f))
    combined = ''.join(hashes).encode()
    return hashlib.md5(combined).hexdigest()

# Чтение и обработка JSON
def load_block_dictionary(json_path):
    start_time = time.time()
    print("Загрузка словаря блоков...")
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Файл {json_path} не найден.")
    with open(json_path, 'r') as f:
        block_dict = json.load(f)

    token_dict = {}
    for key, entry in block_dict.items():
        key = int(key)
        block_state = entry["name"]
        parts = block_state.replace(']', '').split('[')
        block_name = parts[0]
        states = parts[1].split(',') if len(parts) > 1 else []
        tokens = [block_name] + states
        token_dict[key] = tokens

    missing_keys = set(map(str, block_dict.keys())) - set(map(str, token_dict.keys()))
    if missing_keys:
        print(f"Предупреждение: Отсутствуют ключи в token_dict: {missing_keys}")

    print(f"Словарь блоков загружен за {time.time() - start_time:.2f}s")
    return token_dict, block_dict

# Загрузка модели Word2Vec и связанных данных
def load_word2vec_model(model_path="word2vec.model", token_dict_path="token_dict.json", dataset_hash_path="dataset_hash.json"):
    """Загружает модель Word2Vec и связанные данные."""
    if not all(os.path.exists(f) for f in [model_path, token_dict_path, dataset_hash_path]):
        raise FileNotFoundError("Один или несколько файлов (word2vec.model, token_dict.json, dataset_hash.json) не найдены.")

    with open(dataset_hash_path, 'r') as f:
        saved_hash = json.load(f)['hash']
    model = Word2Vec.load(model_path)
    with open(token_dict_path, 'r') as f:
        token_dict = {int(k): v for k, v in json.load(f).items()}
    return saved_hash, model, token_dict

def structure_to_embeddings(structure, token_dict, w2v_model, embedding_dim=16):
    embedding_structure = np.zeros((structure.shape[0], structure.shape[1], structure.shape[2], embedding_dim))
    for x in range(structure.shape[0]):
        for y in range(structure.shape[1]):
            for z in range(structure.shape[2]):
                key = structure[x, y, z].item()
                if key in token_dict:
                    tokens = token_dict[key]
                    embedding = np.zeros(embedding_dim)
                    count = 0
                    for token in tokens:
                        if token in w2v_model.wv:
                            embedding += w2v_model.wv[token]
                            count += 1
                    if count > 0:
                        embedding_structure[x, y, z] = embedding / count
    return embedding_structure

# Загрузка модели VQGAN
class VQGANFullRes(torch.nn.Module):
    def __init__(self, grid_size=32, embedding_dim=16, codebook_size=512):
        super().__init__()
        self.grid_size = grid_size
        self.embedding_dim = embedding_dim
        self.codebook_size = codebook_size
        self.latent_dim = grid_size // 8
        self.encoder = torch.nn.Sequential(
            torch.nn.Conv3d(embedding_dim, 16, kernel_size=3, stride=2, padding=1), torch.nn.ReLU(),
            torch.nn.Conv3d(16, 32, kernel_size=3, stride=2, padding=1), torch.nn.ReLU(),
            torch.nn.Conv3d(32, embedding_dim, kernel_size=3, stride=2, padding=1), torch.nn.ReLU(),
            torch.nn.Conv3d(embedding_dim, embedding_dim, kernel_size=3, stride=1, padding=1),
        )
        self.embedding = torch.nn.Embedding(codebook_size, embedding_dim)
        self.decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose3d(embedding_dim, embedding_dim, kernel_size=3, stride=1, padding=1), torch.nn.ReLU(),
            torch.nn.ConvTranspose3d(embedding_dim, 32, kernel_size=3, stride=2, padding=1, output_padding=1), torch.nn.ReLU(),
            torch.nn.ConvTranspose3d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), torch.nn.ReLU(),
            torch.nn.ConvTranspose3d(16, embedding_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
        )
        self.discriminator = torch.nn.Sequential(
            torch.nn.Conv3d(embedding_dim, 32, kernel_size=4, stride=2, padding=1), torch.nn.LeakyReLU(0.2),
            torch.nn.Conv3d(32, 64, kernel_size=4, stride=2, padding=1), torch.nn.LeakyReLU(0.2),
            torch.nn.Conv3d(64, 1, kernel_size=4, stride=1, padding=0), torch.nn.Sigmoid(),
        )

    def quantize(self, z):
        z_flattened = z.permute(0, 2, 3, 4, 1).reshape(-1, self.embedding_dim)
        distances = torch.cdist(z_flattened, self.embedding.weight)
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z.shape)
        return z_q, encoding_indices

    def forward(self, x):
        z = self.encoder(x)
        z_q, indices = self.quantize(z)
        x_recon = self.decoder(z_q)
        return x_recon, indices

    def discriminate(self, x):
        return self.discriminator(x)

# Загрузка модели Diffusion
class DiscreteDiffusion(torch.nn.Module):
    def __init__(self, codebook_size=512, timesteps=500, embedding_dim=16, latent_dim=2):
        super().__init__()
        self.codebook_size = codebook_size
        self.timesteps = timesteps
        self.latent_dim = latent_dim
        self.embedding = torch.nn.Embedding(codebook_size, embedding_dim)
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=4, dim_feedforward=512, batch_first=True),
            num_layers=4
        )
        self.output_layer = torch.nn.Linear(embedding_dim, codebook_size)

    def forward(self, indices, t):
        batch_size, seq_len = indices.shape
        mask = torch.rand(batch_size, seq_len, device=indices.device) < (t / self.timesteps).view(-1, 1)
        noisy_indices = torch.where(mask, torch.randint(0, self.codebook_size, indices.shape, device=indices.device), indices)
        x = self.embedding(noisy_indices)
        x = self.transformer(x)
        pred = self.output_layer(x)
        return pred

    def sample(self, batch_size, seq_len, device, block_weights=None):
        if block_weights is None:
            indices = torch.randint(0, self.codebook_size, (batch_size, seq_len), device=device)
        else:
            num_blocks = block_weights.size(0)
            indices = torch.zeros(batch_size, seq_len, dtype=torch.long, device=device)
            print(f"block_weights shape: {block_weights.shape}, Min: {torch.min(block_weights)}, Max: {torch.max(block_weights)}")
            if torch.isnan(block_weights).any() or torch.isinf(block_weights).any():
                print("Warning: block_weights contains NaN or Inf values")
                block_weights = torch.nan_to_num(block_weights, nan=0.0, posinf=0.0, neginf=0.0)
            # Сокращаем block_weights до codebook_size с сохранением пропорций
            if num_blocks > self.codebook_size:
                weights_sum = torch.sum(block_weights)
                if weights_sum == 0:
                    weights_sum = 1.0
                normalized_weights = block_weights / weights_sum
                sampled_indices = torch.multinomial(normalized_weights, num_samples=seq_len * batch_size, replacement=True)
                mapping = torch.linspace(0, self.codebook_size - 1, steps=num_blocks).long().to(device)  # Перенос на GPU
                for i in range(batch_size):
                    start_idx = i * seq_len
                    end_idx = (i + 1) * seq_len
                    indices[i] = mapping[sampled_indices[start_idx:end_idx] % self.codebook_size]
            else:
                for i in range(batch_size):
                    indices[i] = torch.multinomial(block_weights, num_samples=seq_len, replacement=True) % self.codebook_size
        print(f"Generated indices shape: {indices.shape}, Min: {torch.min(indices)}, Max: {torch.max(indices)}")
        return indices

# Датасет
class MinecraftHouseDataset:
    def __init__(self, data_dir, grid_size=32, global_blocklist_file=None, embedding_dim=16, cache_dir="cache"):
        self.data_dir = data_dir
        self.grid_size = grid_size
        self.embedding_dim = embedding_dim
        self.cache_dir = cache_dir
        self.files = []

        start_time = time.time()
        print(f"Сканирование директории {data_dir}...")
        for root, _, filenames in os.walk(data_dir):
            for filename in filenames:
                if filename.endswith('_normalized.npy'):
                    self.files.append(os.path.join(root, filename))
        print(f"Сканирование завершено за {time.time() - start_time:.2f}s, найдено {len(self.files)} файлов")

        if not self.files:
            raise ValueError(f"Не найдено .npy файлов в {data_dir}")

        if global_blocklist_file and not os.path.exists(global_blocklist_file):
            raise FileNotFoundError(f"Файл {global_blocklist_file} не найден.")
        if global_blocklist_file:
            self.token_dict, self.block_dict = load_block_dictionary(global_blocklist_file)
        else:
            raise ValueError("Требуется global_blocklist_file")

        self.block_keys = [int(k) for k in self.block_dict.keys()]  # Восстановление атрибута block_keys

        dataset_hash = compute_dataset_hash(self.files)
        saved_hash, saved_model, saved_token_dict = load_word2vec_model()

        if saved_hash != dataset_hash:
            raise ValueError("Хэш набора данных не соответствует сохранённому. Проверьте данные или выполните обучение заново.")
        if not saved_model or not saved_token_dict:
            raise ValueError("Модель Word2Vec или token_dict не загружены. Убедитесь, что файлы существуют.")

        print("Загрузка сохранённой модели Word2Vec...")
        self.w2v_model = saved_model
        self.token_dict = saved_token_dict

        if not os.path.exists("block_embeddings.pt"):
            raise FileNotFoundError("Файл block_embeddings.pt не найден. Убедитесь, что он существует.")
        self.block_embeddings = torch.load("block_embeddings.pt", map_location="cpu")
        self.kdtree = KDTree(self.block_embeddings.numpy())

# Генерация дома
def generate_house(vqgan_full_res, diffusion, dataset, grid_size=32, device=None):
    start_time = time.time()
    print("Генерация дома...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используется устройство: {device}")
    vqgan_full_res = vqgan_full_res.to(device)
    diffusion = diffusion.to(device)
    latent_size = vqgan_full_res.latent_dim ** 3

    # Получение весов блоков из тренировочных данных
    block_counts = {}
    for file in dataset.files:
        structure = np.load(file).astype(np.int64)
        unique, counts = np.unique(structure, return_counts=True)
        for block_id, count in zip(unique, counts):
            block_counts[block_id] = block_counts.get(block_id, 0) + count
    total_voxels = sum(block_counts.values())
    block_weights = torch.zeros(len(dataset.block_keys), device=device)
    for i, key in enumerate(dataset.block_keys):
        block_weights[i] = block_counts.get(key, 0) / total_voxels if total_voxels > 0 else 1.0 / len(dataset.block_keys)

    indices = diffusion.sample(batch_size=1, seq_len=latent_size, device=device, block_weights=block_weights)
    print(f"Generated indices shape: {indices.shape}, Min: {torch.min(indices)}, Max: {torch.max(indices)}")
    z_q = vqgan_full_res.embedding(indices).view(1, vqgan_full_res.embedding_dim, vqgan_full_res.latent_dim, vqgan_full_res.latent_dim, vqgan_full_res.latent_dim)
    with torch.no_grad(), autocast(device_type=device.type):
        recon = vqgan_full_res.decoder(z_q)

    # Отладка: проверка формы и значений
    print(f"recon shape: {recon.shape}")
    if torch.isnan(recon).any() or torch.isinf(recon).any():
        print("Warning: recon contains NaN or Inf values")
        recon = torch.nan_to_num(recon, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"recon min: {torch.min(recon)}, max: {torch.max(recon)}")

    recon = recon.squeeze(0).permute(1, 2, 3, 0).to(torch.float32)  # Приведение к float32
    expected_size = grid_size * grid_size * grid_size * dataset.embedding_dim
    if recon.numel() != expected_size:
        print(f"Warning: recon size {recon.numel()} does not match expected {expected_size}")
        recon = recon.view(grid_size, grid_size, grid_size, dataset.embedding_dim)

    recon_flat = recon.view(-1, dataset.embedding_dim).cpu().numpy()
    _, indices = dataset.kdtree.query(recon_flat)
    best_keys = np.array(dataset.block_keys)[indices]
    best_keys = np.array([k for k in best_keys if k in dataset.block_keys])
    if len(best_keys) != grid_size * grid_size * grid_size:
        best_keys = np.pad(best_keys, (0, grid_size * grid_size * grid_size - len(best_keys)), mode='constant', constant_values=0)
    generated_house = best_keys.reshape(grid_size, grid_size, grid_size)

    print(f"Дом сгенерирован за {time.time() - start_time:.2f}с")
    return generated_house

# Основной код
if __name__ == "__main__":
    data_dir = "/content/Data/Output"
    global_blocklist_file = "/content/global_block_list_transformed.json"
    grid_size = 32
    device = None  # Автоматический выбор устройства
    model_vqgan_path = "vqgan_full_res_model.pth"
    model_diffusion_path = "diffusion_model.pth"

    # Проверка существования файлов моделей
    if not os.path.exists(model_vqgan_path):
        raise FileNotFoundError(f"Файл {model_vqgan_path} не найден.")
    if not os.path.exists(model_diffusion_path):
        raise FileNotFoundError(f"Файл {model_diffusion_path} не найден.")

    # Загрузка датасета
    dataset = MinecraftHouseDataset(data_dir, grid_size, global_blocklist_file)

    # Загрузка моделей
    print("Инициализация модели VQGAN...")
    vqgan_full_res = VQGANFullRes(grid_size=grid_size, embedding_dim=16, codebook_size=512)
    for name, param in vqgan_full_res.named_parameters():
        print(f"Layer {name}: Shape {param.shape}, Min {torch.min(param)}, Max {torch.max(param)}")
    vqgan_full_res.load_state_dict(torch.load(model_vqgan_path, map_location=torch.device('cpu')))
    vqgan_full_res.eval()

    print("Инициализация модели Diffusion...")
    diffusion = DiscreteDiffusion(codebook_size=512, timesteps=500, embedding_dim=16, latent_dim=2)
    for name, param in diffusion.named_parameters():
        print(f"Layer {name}: Shape {param.shape}, Min {torch.min(param)}, Max {torch.max(param)}")
    diffusion.load_state_dict(torch.load(model_diffusion_path, map_location=torch.device('cpu')))
    diffusion.eval()

    # Генерация дома
    generated_house = generate_house(vqgan_full_res, diffusion, dataset, grid_size, device)
    np.save("generated_house.npy", generated_house)
    print("Сгенерированный дом сохранён в generated_house.npy")